# NILM Phase 2 — Ablation: CNN + BiLSTM (No Knowledge Vector)
**Dataset:** UK-DALE 2017, House 1
**Appliances:** Dishwasher, Washing Machine, Laptop, Fridge, Office Lamp 1

**Purpose:** This is a controlled ablation of `NILM_Phase2_CNN_Knowledge.ipynb`. Everything
about the data pipeline (loading, resampling, median filtering, ON/OFF labeling, the
block-stratified day-cycled split, and window construction) is **identical** to the
knowledge-vector version, so results are directly comparable.

**What's removed vs. the knowledge version:**
1. The 12-dim hand-engineered knowledge vector (mean/max/std/RMS/power-jump/etc.) is no
   longer computed or fed into the model.
2. The knowledge-extractor Dense branch is removed.
3. The attention-gated knowledge fusion (`knowledge_gate` -> `gated_context` -> concat) is
   removed — the temporal-attention context vector goes straight to the shared dense layers.
4. The knowledge-vector reconstruction auxiliary head/loss is removed (there's no knowledge
   vector left to reconstruct).

**What's kept identical:** 1D-CNN branch -> BiLSTM -> temporal attention pooling -> shared
dense layers -> state (multi-label ON/OFF) and duty-cycle regression heads -> post-hoc
minimum-dwell-time correction. Same window size/stride, same thresholds, same class
weights, same split, same training config (epochs/batch size/callbacks).

Run this notebook on the same data and compare its test-set precision/recall/F1 and
duration-estimation error against the knowledge-vector notebook's results to see whether
the knowledge vector is actually contributing anything.


## 1. Install Dependencies

In [ ]:
!pip install -q tensorflow scikit-learn scipy pandas numpy matplotlib seaborn

## 2. Import Libraries

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import medfilt, find_peaks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              classification_report, confusion_matrix)
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## 3. Mount Google Drive & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    "house_dir": "/content/drive/MyDrive/SEM7PROJ/house_1",  # adjust to your folder structure
    "checkpoint_dir": "/content/drive/MyDrive/SEM7PROJ/checkpoints_phase2_noknowledge",
    "sample_period": "6s",
    "window_size": 100,       # ~10 minutes at 6s/sample
    "window_stride": 20,
    "median_kernel": 5,
    "on_thresholds": {
        "dishwasher": 10,
        "washing machine": 20,
        "laptop": 15,
        "fridge": 20,
        "office lamp 1": 5,
    },
    # Minimum consecutive samples above threshold to count as a real ON event (denoise).
    "min_on_samples": {
        "dishwasher": 20,
        "washing machine": 10,
        "laptop": 5,
        "fridge": 5,
        "office lamp 1": 3,
    },
    "label_rule": {
        "dishwasher": "majority",
        "washing machine": "majority",
        "laptop": "majority",
        "fridge": "majority",
        "office lamp 1": "majority",
    },
    "appliances": ["dishwasher", "washing machine", "laptop", "fridge", "office lamp 1"],
    "batch_size": 64,
    "epochs": 50,
    "min_dwell_windows": 2,   # post-hoc correction: flip isolated runs shorter than this
    "duty_loss_weight": 1.0,
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
print("Config ready. House directory:", CONFIG["house_dir"])

Mounted at /content/drive
Config ready. House directory: /content/drive/MyDrive/SEM7PROJ/house_1


## 4. Dataset Loading

In [ ]:
def load_labels(house_dir):
    labels_path = os.path.join(house_dir, "labels.dat")
    labels = {}
    with open(labels_path, "r") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                ch_num, name = parts
                labels[int(ch_num)] = name.strip().lower().replace("_", " ")
    return labels

def load_channel(house_dir, channel_num):
    path_dat = os.path.join(house_dir, f"channel_{channel_num}.dat")
    path_csv = os.path.join(house_dir, f"channel_{channel_num}.csv")
    if os.path.exists(path_dat):
        df = pd.read_csv(path_dat, sep=" ", header=None, names=["timestamp", "power"])
    elif os.path.exists(path_csv):
        df = pd.read_csv(path_csv, header=None, names=["timestamp", "power"])
    else:
        raise FileNotFoundError(f"No data file found for channel {channel_num} in {house_dir}")
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df = df.set_index("timestamp").sort_index()
    df["power"] = pd.to_numeric(df["power"], errors="coerce")
    return df

labels_map = load_labels(CONFIG["house_dir"])
print(f"Found {len(labels_map)} channels in House 1.")

Found 53 channels in House 1.


## 5. Appliance Selection, Shared-Origin Resampling & Aggregate Signal

In [ ]:
def find_channel_for_appliance(labels_map, appliance_name):
    appliance_name = appliance_name.lower()
    candidates = [ch for ch, name in labels_map.items()
                  if appliance_name in name or name in appliance_name]
    if not candidates:
        words = appliance_name.split()
        candidates = [ch for ch, name in labels_map.items() if all(w in name for w in words)]
    if not candidates:
        raise ValueError(f"No matching channel found for appliance '{appliance_name}'.")
    return candidates[0]

selected_channels = {}
appliance_data = {}
for app in CONFIG["appliances"]:
    ch = find_channel_for_appliance(labels_map, app)
    selected_channels[app] = ch
    df = load_channel(CONFIG["house_dir"], ch)
    appliance_data[app] = df
    print(f"{app:20s} -> channel {ch} | {len(df)} samples, {df.index.min()} to {df.index.max()}")

def resample_series(df, period, origin):
    return df["power"].resample(period, origin=origin).mean()

shared_origin = min(df.index.min() for df in appliance_data.values())
print("\nShared resample origin:", shared_origin)

resampled = {app: resample_series(df, CONFIG["sample_period"], shared_origin)
             for app, df in appliance_data.items()}

combined = pd.DataFrame(resampled).dropna(how="all")
max_ffill = int(60 / 6)
combined = combined.ffill(limit=max_ffill)
combined["aggregate"] = combined[CONFIG["appliances"]].sum(axis=1, skipna=True)

print("\nSynthetic aggregate created from:", CONFIG["appliances"])
print("Total samples:", len(combined))

dishwasher           -> channel 6 | 19819392 samples, 2012-11-09 22:28:18 to 2017-04-26 17:35:30
washing machine      -> channel 5 | 19555935 samples, 2012-11-09 22:28:18 to 2017-04-26 17:35:24
laptop               -> channel 4 | 4539118 samples, 2012-11-16 11:29:01 to 2017-04-26 17:34:48
fridge               -> channel 12 | 19381298 samples, 2012-12-14 22:21:32 to 2017-04-26 17:32:51
office lamp 1        -> channel 48 | 18179281 samples, 2013-04-11 19:08:55 to 2017-04-26 17:35:12

Shared resample origin: 2012-11-09 22:28:18

Synthetic aggregate created from: ['dishwasher', 'washing machine', 'laptop', 'fridge', 'office lamp 1']
Total samples: 22736289


## 6. Preprocessing (median filtering, missing-value handling)

In [ ]:
data = combined.dropna(subset=CONFIG["appliances"] + ["aggregate"]).copy()

def median_filter_series(series, kernel_size):
    if kernel_size % 2 == 0:
        kernel_size += 1
    return pd.Series(medfilt(series.values, kernel_size=kernel_size), index=series.index)

filtered = data.copy()
for col in CONFIG["appliances"] + ["aggregate"]:
    filtered[col] = median_filter_series(data[col], CONFIG["median_kernel"])
filtered[filtered < 0] = 0

print("Preprocessing complete. Rows remaining:", len(filtered))

Preprocessing complete. Rows remaining: 3930087


In [ ]:
# ============================================================
# 5-Fold Cross-Validation (day-grouped) for CNN+BiLSTM (No Knowledge)
# Self-contained: includes model architecture, doesn't depend on
# cell order elsewhere in the notebook.
# ============================================================
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np, os, json

# ---- 0. Model architecture (same as your Section 11) ----
def build_cnn_branch(window_size):
    seq_input = layers.Input(shape=(window_size, 1), name="aggregate_sequence")
    x = layers.Conv1D(32, 5, padding="same", activation="relu")(seq_input)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return seq_input, x

def weighted_bce(pos_weights):
    pw = tf.constant(pos_weights, dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        bce = -(pw * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce)
    return loss_fn

def build_model(window_size, n_appliances, pos_weights, duty_weight):
    seq_input, cnn_features = build_cnn_branch(window_size)

    bilstm_out = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(cnn_features)
    bilstm_out = layers.Dropout(0.3)(bilstm_out)

    attn_scores = layers.Dense(1, activation="tanh")(bilstm_out)
    attn_weights = layers.Softmax(axis=1)(attn_scores)
    context = layers.Multiply()([bilstm_out, attn_weights])
    context = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="temporal_attention_pool")(context)

    shared = layers.Dense(64, activation="relu")(context)
    shared = layers.Dropout(0.3)(shared)
    shared = layers.Dense(32, activation="relu")(shared)

    state_out = layers.Dense(n_appliances, activation="sigmoid", name="state")(shared)
    duty_out = layers.Dense(n_appliances, activation="sigmoid", name="duty")(shared)

    model = models.Model(inputs=seq_input, outputs=[state_out, duty_out],
                          name="CNN_BiLSTM_Attention_NoKnowledge")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={"state": weighted_bce(pos_weights), "duty": "mse"},
        loss_weights={"state": 1.0, "duty": duty_weight},
        metrics={"state": [tf.keras.metrics.BinaryAccuracy(name="acc"),
                            tf.keras.metrics.Precision(name="precision"),
                            tf.keras.metrics.Recall(name="recall")]}
    )
    return model

# ---- 0b. Dwell-time correction (same as your Section 17) ----
def apply_min_dwell_correction(state_sequence, min_dwell):
    corrected = state_sequence.copy()
    n = len(corrected)
    i = 0
    while i < n:
        j = i
        while j < n and corrected[j] == corrected[i]:
            j += 1
        run_length = j - i
        if run_length < min_dwell and 0 < i and j < n:
            if corrected[i - 1] == corrected[j] and corrected[i] != corrected[j]:
                corrected[i:j] = corrected[j]
        i = j
    return corrected

# ---- 1–2. Windows + day_id: you've already run this — X_all_seq, Y_all_state,
#            Y_all_duty, day_id_all should still be in memory. If not, re-run
#            that block before continuing. ----

# ---- 3. 5-Fold CV loop ----
N_FOLDS = 5
KFOLD_DIR = os.path.join(os.path.dirname(CONFIG["checkpoint_dir"]), "checkpoints_kfold_noknowledge")
os.makedirs(KFOLD_DIR, exist_ok=True)

gkf = GroupKFold(n_splits=N_FOLDS)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_all_seq, Y_all_state, groups=day_id_all), start=1):
    print(f"\n{'='*60}\nFold {fold}/{N_FOLDS}\n{'='*60}")

    X_tr, X_val = X_all_seq[train_idx], X_all_seq[val_idx]
    Ys_tr, Ys_val = Y_all_state[train_idx], Y_all_state[val_idx]
    Yd_tr, Yd_val = Y_all_duty[train_idx], Y_all_duty[val_idx]

    print(f"train windows: {len(train_idx)} | val windows: {len(val_idx)}")
    print(f"train days: {len(np.unique(day_id_all[train_idx]))} | val days: {len(np.unique(day_id_all[val_idx]))}")

    seq_scaler = MinMaxScaler()
    X_tr_s = seq_scaler.fit_transform(X_tr.reshape(-1, 1)).reshape(X_tr.shape)[..., np.newaxis]
    X_val_s = seq_scaler.transform(X_val.reshape(-1, 1)).reshape(X_val.shape)[..., np.newaxis]

    pos_counts = Ys_tr.sum(axis=0).astype(np.float32)
    neg_counts = (Ys_tr.shape[0] - pos_counts).astype(np.float32)
    pos_weights_arr = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 15.0)

    tf.keras.backend.clear_session()
    model = build_model(CONFIG["window_size"], len(CONFIG["appliances"]),
                         pos_weights_arr, CONFIG["duty_loss_weight"])

    fold_ckpt = os.path.join(KFOLD_DIR, f"fold_{fold}_best.keras")
    callback_list = [
        callbacks.ModelCheckpoint(fold_ckpt, monitor="val_loss", save_best_only=True, verbose=0),
        callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=0),
    ]

    model.fit(
        X_tr_s, {"state": Ys_tr, "duty": Yd_tr},
        validation_data=(X_val_s, {"state": Ys_val, "duty": Yd_val}),
        epochs=CONFIG["epochs"], batch_size=CONFIG["batch_size"],
        callbacks=callback_list, verbose=1
    )

    state_prob, duty_pred = model.predict(X_val_s, verbose=0)
    state_pred = (state_prob > 0.5).astype(int)

    state_pred_corrected = state_pred.copy()
    for i in range(len(CONFIG["appliances"])):
        state_pred_corrected[:, i] = apply_min_dwell_correction(
            state_pred[:, i], CONFIG["min_dwell_windows"]
        )

    per_appliance = {}
    for i, app in enumerate(CONFIG["appliances"]):
        p = precision_score(Ys_val[:, i], state_pred_corrected[:, i], zero_division=0)
        r = recall_score(Ys_val[:, i], state_pred_corrected[:, i], zero_division=0)
        f1 = f1_score(Ys_val[:, i], state_pred_corrected[:, i], zero_division=0)
        acc = accuracy_score(Ys_val[:, i], state_pred_corrected[:, i])
        per_appliance[app] = {"precision": p, "recall": r, "f1": f1, "accuracy": acc}
        print(f"  {app:20s} | Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f} | Acc: {acc:.3f}")

    macro_p = precision_score(Ys_val, state_pred_corrected, average="macro", zero_division=0)
    macro_r = recall_score(Ys_val, state_pred_corrected, average="macro", zero_division=0)
    macro_f1 = f1_score(Ys_val, state_pred_corrected, average="macro", zero_division=0)
    macro_acc = accuracy_score(Ys_val, state_pred_corrected)

    print(f"\n  MACRO  -> Precision: {macro_p:.3f} | Recall: {macro_r:.3f} | F1: {macro_f1:.3f} | Acc: {macro_acc:.3f}")

    fold_results.append({
        "fold": fold, "n_train": int(len(train_idx)), "n_val": int(len(val_idx)),
        "per_appliance": per_appliance,
        "macro_precision": float(macro_p), "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1), "macro_accuracy": float(macro_acc),
    })

# ---- 4. Summary ----
print(f"\n{'='*60}\n5-Fold CV Summary (macro, over appliances)\n{'='*60}")
print(f"{'Fold':6s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Accuracy':>10s}")
for r in fold_results:
    print(f"{r['fold']:<6d} {r['macro_precision']:>10.4f} {r['macro_recall']:>10.4f} "
          f"{r['macro_f1']:>10.4f} {r['macro_accuracy']:>10.4f}")

f1s, precs, recs, accs = ([r[k] for r in fold_results] for k in
                           ("macro_f1", "macro_precision", "macro_recall", "macro_accuracy"))
print(f"\n{'Mean':6s} {np.mean(precs):>10.4f} {np.mean(recs):>10.4f} {np.mean(f1s):>10.4f} {np.mean(accs):>10.4f}")
print(f"{'Std':6s} {np.std(precs):>10.4f} {np.std(recs):>10.4f} {np.std(f1s):>10.4f} {np.std(accs):>10.4f}")

print(f"\nPer-appliance F1 across folds:")
for app in CONFIG["appliances"]:
    app_f1s = [r["per_appliance"][app]["f1"] for r in fold_results]
    print(f"  {app:20s} F1: {np.mean(app_f1s):.4f} ± {np.std(app_f1s):.4f}")

results_path = os.path.join(KFOLD_DIR, "kfold_results.json")
with open(results_path, "w") as f:
    json.dump(fold_results, f, indent=2)
print(f"\nSaved fold results to {results_path}")


Fold 1/5
train windows: 157200 | val windows: 39300
train days: 725 | val days: 181
Epoch 1/50
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 66s 23ms/step - duty_loss: 0.0519 - loss: 0.3717 - state_acc: 0.8968 - state_loss: 0.3199 - state_precision: 0.7364 - state_recall: 0.9264 - val_duty_loss: 0.1765 - val_loss: 1.4171 - val_state_acc: 0.6729 - val_state_loss: 1.2387 - val_state_precision: 0.3637 - val_state_recall: 0.5260 - learning_rate: 0.0010
Epoch 2/50
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 57s 23ms/step - duty_loss: 0.0381 - loss: 0.2851 - state_acc: 0.9221 - state_loss: 0.2470 - state_precision: 0.7892 - state_recall: 0.9471 - val_duty_loss: 0.0637 - val_loss: 0.4453 - val_state_acc: 0.8450 - val_state_loss: 0.3810 - val_state_precision: 0.6075 - val_state_recall: 0.9581 - learning_rate: 0.0010
Epoch 3/50
2457/2457 ━━━━━━━━━━━━━━━━━━━━ 56s 23ms/step - duty_loss: 0.0343 - loss: 0.2539 - state_acc: 0.9302 - state_loss: 0.2196 - state_precision: 0.8085 - state_recall: 0.9511 - val_duty_loss: 0.0480 - 

bold text## 7. Persistent ON/OFF State & Block-Stratified Split
Same as the knowledge-vector notebook: a persistence rule denoises brief power blips
before labeling, and a day-cycled block split (7 train : 1 val : 1 test days, repeating)
guarantees every appliance has representative coverage in every split.

In [ ]:
def compute_on_mask(power, threshold, min_samples):
    raw_on = power > threshold
    if min_samples <= 1:
        return raw_on
    groups = (raw_on != raw_on.shift()).cumsum()
    run_lengths = raw_on.groupby(groups).transform("size")
    return raw_on & (run_lengths >= min_samples)

for app in CONFIG["appliances"]:
    filtered[f"{app}_on"] = compute_on_mask(
        filtered[app], CONFIG["on_thresholds"][app], CONFIG["min_on_samples"][app]
    )

filtered["date"] = filtered.index.normalize()
unique_days = filtered["date"].drop_duplicates().sort_values().reset_index(drop=True)
day_split = []
for i in range(len(unique_days)):
    r = i % 10
    day_split.append("train" if r < 7 else ("val" if r == 7 else "test"))
day_assignment = pd.Series(day_split, index=unique_days)
filtered["split"] = filtered["date"].map(day_assignment)

train_df = filtered[filtered["split"] == "train"].drop(columns=["date", "split"])
val_df = filtered[filtered["split"] == "val"].drop(columns=["date", "split"])
test_df = filtered[filtered["split"] == "test"].drop(columns=["date", "split"])

print(f"Rows -> train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")
print("\nAppliance ON-event coverage per split:")
for app in CONFIG["appliances"]:
    counts = {s: int(d[f"{app}_on"].sum()) for s, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
    flag = " <-- ZERO EVENTS" if 0 in counts.values() else ""
    print(f"  {app:20s} {counts}{flag}")

Rows -> train: 2739242 | val: 417176 | test: 773669

Appliance ON-event coverage per split:
  dishwasher           {'train': 74214, 'val': 11831, 'test': 18415}
  washing machine      {'train': 182825, 'val': 31428, 'test': 51911}
  laptop               {'train': 1421844, 'val': 234348, 'test': 407342}
  fridge               {'train': 1228411, 'val': 187106, 'test': 343682}
  office lamp 1        {'train': 492556, 'val': 84517, 'test': 117928}


## 8. Sliding Window Generation — State Labels & Duty-Cycle Targets
**No knowledge vector here.** Each window still gets the two targets that don't depend on
the knowledge vector: (a) a binary state label (via the per-appliance `label_rule`), and
(b) a duty-cycle regression target (the true fraction of the window the appliance was ON).
The 12-dim knowledge vector and its reconstruction target are gone entirely — the model
only ever sees the raw aggregate power window.

In [ ]:
def build_windows(df, window_size, stride, appliances, label_rule):
    agg = df["aggregate"].values
    n_samples = len(agg)

    X_seq, Y_state, Y_duty = [], [], []
    for start in range(0, n_samples - window_size + 1, stride):
        end = start + window_size
        window_agg = agg[start:end]
        if np.isnan(window_agg).any():
            continue

        state_labels, duty_targets = [], []
        for app in appliances:
            on_window = df[f"{app}_on"].values[start:end]
            duty_frac = float(np.mean(on_window))
            if label_rule[app] == "any":
                is_on = int(on_window.any())
            else:
                is_on = int(duty_frac > 0.5)
            state_labels.append(is_on)
            duty_targets.append(duty_frac)

        X_seq.append(window_agg)
        Y_state.append(state_labels)
        Y_duty.append(duty_targets)

    return (np.array(X_seq), np.array(Y_state), np.array(Y_duty, dtype=np.float32))

X_train_seq, Y_train_state, Y_train_duty = build_windows(
    train_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])
X_val_seq, Y_val_state, Y_val_duty = build_windows(
    val_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])
X_test_seq, Y_test_state, Y_test_duty = build_windows(
    test_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])

print("Train:", X_train_seq.shape, Y_train_state.shape, Y_train_duty.shape)
print("Val:  ", X_val_seq.shape, Y_val_state.shape, Y_val_duty.shape)
print("Test: ", X_test_seq.shape, Y_test_state.shape, Y_test_duty.shape)

Train: (136958, 100) (136958, 5) (136958, 5)
Val:   (20854, 100) (20854, 5) (20854, 5)
Test:  (38679, 100) (38679, 5) (38679, 5)


## 9. Dataset Preparation — Scaling (fit on TRAIN only)

In [ ]:
seq_scaler = MinMaxScaler()
X_train_seq_s = seq_scaler.fit_transform(X_train_seq.reshape(-1, 1)).reshape(X_train_seq.shape)[..., np.newaxis]
X_val_seq_s = seq_scaler.transform(X_val_seq.reshape(-1, 1)).reshape(X_val_seq.shape)[..., np.newaxis]
X_test_seq_s = seq_scaler.transform(X_test_seq.reshape(-1, 1)).reshape(X_test_seq.shape)[..., np.newaxis]

with open(os.path.join(CONFIG["checkpoint_dir"], "scalers.pkl"), "wb") as f:
    pickle.dump({"seq_scaler": seq_scaler, "config": CONFIG}, f)

print("Scaled shapes -> seq:", X_train_seq_s.shape)

Scaled shapes -> seq: (136958, 100, 1)


## 10. Per-Appliance Class Weights (for the state-classification loss)

In [ ]:
pos_counts = Y_train_state.sum(axis=0).astype(np.float32)
neg_counts = (Y_train_state.shape[0] - pos_counts).astype(np.float32)
pos_weights_arr = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 15.0)

print("Per-appliance positive-class weights:")
for app, w, pos in zip(CONFIG["appliances"], pos_weights_arr, pos_counts):
    print(f"  {app:20s} weight={w:6.2f}  (positive windows in train: {int(pos)})")

Per-appliance positive-class weights:
  dishwasher           weight= 15.00  (positive windows in train: 3797)
  washing machine      weight= 12.82  (positive windows in train: 9907)
  laptop               weight=  1.00  (positive windows in train: 71538)
  fridge               weight=  1.23  (positive windows in train: 61393)
  office lamp 1        weight=  4.56  (positive windows in train: 24652)


## 11. Model Architecture — CNN + BiLSTM + Attention (No Knowledge Fusion)
Same CNN branch and same BiLSTM + temporal-attention pooling as the knowledge-vector
notebook, but the pooled attention context now goes **directly** to the shared dense
layers — no knowledge embedding, no gate, no concatenation, no reconstruction head.
Only two task heads remain: state (multi-label ON/OFF) and duty-cycle regression.

In [ ]:
def build_cnn_branch(window_size):
    seq_input = layers.Input(shape=(window_size, 1), name="aggregate_sequence")
    x = layers.Conv1D(32, 5, padding="same", activation="relu")(seq_input)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return seq_input, x  # (batch, window_size, 64) sequence of local CNN features

def build_model(window_size, n_appliances, pos_weights, duty_weight):
    seq_input, cnn_features = build_cnn_branch(window_size)

    # BiLSTM over the CNN feature sequence
    bilstm_out = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(cnn_features)
    bilstm_out = layers.Dropout(0.3)(bilstm_out)

    # Temporal attention: learn a weight per timestep, pool into one context vector
    attn_scores = layers.Dense(1, activation="tanh")(bilstm_out)          # (batch, T, 1)
    attn_weights = layers.Softmax(axis=1)(attn_scores)                    # (batch, T, 1)
    context = layers.Multiply()([bilstm_out, attn_weights])
    context = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="temporal_attention_pool")(context)  # (batch, 64)

    # No knowledge fusion: the attention context goes straight to the shared dense layers.
    shared = layers.Dense(64, activation="relu")(context)
    shared = layers.Dropout(0.3)(shared)
    shared = layers.Dense(32, activation="relu")(shared)

    # Task heads (state + duty only — no reconstruction head, there's no knowledge
    # vector to reconstruct)
    state_out = layers.Dense(n_appliances, activation="sigmoid", name="state")(shared)
    duty_out = layers.Dense(n_appliances, activation="sigmoid", name="duty")(shared)

    model = models.Model(inputs=seq_input,
                          outputs=[state_out, duty_out],
                          name="CNN_BiLSTM_Attention_NoKnowledge")

    def weighted_bce(pos_weights):
        pw = tf.constant(pos_weights, dtype=tf.float32)
        def loss_fn(y_true, y_pred):
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
            y_true = tf.cast(y_true, tf.float32)
            bce = -(pw * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
            return tf.reduce_mean(bce)
        return loss_fn

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={"state": weighted_bce(pos_weights), "duty": "mse"},
        loss_weights={"state": 1.0, "duty": duty_weight},
        metrics={"state": [tf.keras.metrics.BinaryAccuracy(name="acc"),
                            tf.keras.metrics.Precision(name="precision"),
                            tf.keras.metrics.Recall(name="recall")]}
    )
    return model

model = build_model(CONFIG["window_size"], len(CONFIG["appliances"]),
                     pos_weights_arr, CONFIG["duty_loss_weight"])
model.summary()

Model: "CNN_BiLSTM_Attention_NoKnowledge"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ aggregate_sequence  │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 100, 32)   │        192 │ aggregate_sequen… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 100, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 100, 64)   │     10,304 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 100, 64)   │     12,352 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 64)   │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 100, 64)   │     24,832 │ batch_normalizat… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 100, 64)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 100, 1)    │         65 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Softmax)   │ (None, 100, 1)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 100, 64)   │          0 │ dropout[0][0],    │
│                     │                   │            │ softmax[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attention… │ (None, 64)        │          0 │ multiply[0][0]    │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      4,160 │ temporal_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ state (Dense)       │ (None, 5)         │        165 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ duty (Dense)        │ (None, 5)         │        165 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 54,955 (214.67 KB)

 Trainable params: 54,635 (213.42 KB)

 Non-trainable params: 320 (1.25 KB)

## 12. Training Loop (checkpointing, early stopping, LR schedule)

In [ ]:
checkpoint_path = os.path.join(CONFIG["checkpoint_dir"], "best_model.keras")
history_path = os.path.join(CONFIG["checkpoint_dir"], "history.json")

callback_list = [
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1),
]

history = model.fit(
    X_train_seq_s,
    {"state": Y_train_state, "duty": Y_train_duty},
    validation_data=(
        X_val_seq_s,
        {"state": Y_val_state, "duty": Y_val_duty}
    ),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=callback_list,
    verbose=1
)

with open(history_path, "w") as f:
    json.dump(history.history, f)
print("Training complete.")

Epoch 1/50
2140/2140 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - duty_loss: 0.0644 - loss: 0.4515 - state_acc: 0.8747 - state_loss: 0.3871 - state_precision: 0.6969 - state_recall: 0.8793
Epoch 1: val_loss improved from None to 1.19218, saving model to /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2_noknowledge/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2_noknowledge/best_model.keras
2140/2140 ━━━━━━━━━━━━━━━━━━━━ 61s 23ms/step - duty_loss: 0.0511 - loss: 0.3767 - state_acc: 0.8940 - state_loss: 0.3256 - state_precision: 0.7277 - state_recall: 0.9208 - val_duty_loss: 0.1563 - val_loss: 1.1922 - val_state_acc: 0.6280 - val_state_loss: 1.0364 - val_state_precision: 0.3798 - val_state_recall: 0.6367 - learning_rate: 0.0010
Epoch 2/50
2140/2140 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - duty_loss: 0.0417 - loss: 0.3111 - state_acc: 0.9133 - state_loss: 0.2694 - state_precision: 0.7655 - state_recall: 0.9420
Epoch 2: val_loss did not improve fro

## 13. Validation

In [ ]:
val_results = model.evaluate(
    X_val_seq_s,
    {"state": Y_val_state, "duty": Y_val_duty},
    verbose=1
)
print("\nValidation results:")
for name, value in zip(model.metrics_names, val_results):
    print(f"  {name}: {value:.4f}")

652/652 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - duty_loss: 0.0332 - loss: 0.2556 - state_acc: 0.9301 - state_loss: 0.2224 - state_precision: 0.8156 - state_recall: 0.9516

Validation results:
  loss: 0.2556
  compile_metrics: 0.2224
  state_loss: 0.0332
  duty_loss: 0.9301


## 14. Testing

In [ ]:
test_results = model.evaluate(
    X_test_seq_s,
    {"state": Y_test_state, "duty": Y_test_duty},
    verbose=1
)
print("\nTest results:")
for name, value in zip(model.metrics_names, test_results):
    print(f"  {name}: {value:.4f}")

state_prob, duty_pred = model.predict(X_test_seq_s)
state_pred = (state_prob > 0.5).astype(int)

1209/1209 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - duty_loss: 0.0307 - loss: 0.2286 - state_acc: 0.9281 - state_loss: 0.1978 - state_precision: 0.7909 - state_recall: 0.9596

Test results:
  loss: 0.2286
  compile_metrics: 0.1978
  state_loss: 0.0307
  duty_loss: 0.9281
1209/1209 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step


## 15. Saving Checkpoints

In [ ]:
final_model_path = os.path.join(CONFIG["checkpoint_dir"], "final_model.keras")
model.save(final_model_path)
print("Final model saved to:", final_model_path)
print("Best model (lowest val_loss) is saved at:", checkpoint_path)

Final model saved to: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2_noknowledge/final_model.keras
Best model (lowest val_loss) is saved at: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2_noknowledge/best_model.keras


## 16. Resume Training

In [ ]:
def resume_or_build_model():
    if os.path.exists(checkpoint_path):
        print("Existing checkpoint found. Loading model instead of training from scratch...")
        def weighted_bce_loader(pos_weights):
            pw = tf.constant(pos_weights, dtype=tf.float32)
            def loss_fn(y_true, y_pred):
                y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
                y_true = tf.cast(y_true, tf.float32)
                bce = -(pw * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
                return tf.reduce_mean(bce)
            return loss_fn
        loaded_model = tf.keras.models.load_model(
            checkpoint_path,
            custom_objects={"loss_fn": weighted_bce_loader(pos_weights_arr)}
        )
        return loaded_model, True
    else:
        print("No checkpoint found.")
        return build_model(CONFIG["window_size"], len(CONFIG["appliances"]),
                            pos_weights_arr, CONFIG["duty_loss_weight"]), False

resumed_model, was_resumed = resume_or_build_model()
if was_resumed:
    model = resumed_model
    print("Resumed from checkpoint. Re-run Section 12's fit call manually if you want to continue training.")

Existing checkpoint found. Loading model instead of training from scratch...


ValueError: Requested the deserialization of a `Lambda` layer whose `function` is a Python lambda. This carries a potential risk of arbitrary code execution and thus it is disallowed by default. If you trust the source of the artifact, you can override this error by passing `safe_mode=False` to the loading function, or calling `keras.config.enable_unsafe_deserialization().

## 17. Post-Hoc Minimum-Dwell-Time Correction ("Graph Reasoner")
Unchanged from the knowledge-vector notebook: removes implausible single-window ON/OFF
flicker by flipping isolated runs shorter than `min_dwell_windows` to match their
surrounding majority state.

In [ ]:
def apply_min_dwell_correction(state_sequence, min_dwell):
    """state_sequence: 1D binary array (time-ordered) for ONE appliance."""
    corrected = state_sequence.copy()
    n = len(corrected)
    i = 0
    while i < n:
        j = i
        while j < n and corrected[j] == corrected[i]:
            j += 1
        run_length = j - i
        if run_length < min_dwell and 0 < i and j < n:
            if corrected[i - 1] == corrected[j] and corrected[i] != corrected[j]:
                corrected[i:j] = corrected[j]
        i = j
    return corrected

state_pred_corrected = state_pred.copy()
for i, app in enumerate(CONFIG["appliances"]):
    state_pred_corrected[:, i] = apply_min_dwell_correction(state_pred[:, i], CONFIG["min_dwell_windows"])

flips = int((state_pred_corrected != state_pred).sum())
print(f"Dwell-time correction flipped {flips} individual (window, appliance) predictions.")

Dwell-time correction flipped 711 individual (window, appliance) predictions.


## 18. Evaluation
Per-appliance precision/recall/F1 on the state head, before and after the dwell-time
correction, plus duration estimation (predicted vs ground-truth active minutes per
appliance) using the duty-cycle head. Compare these numbers directly against the
knowledge-vector notebook's Section 19 output.

In [ ]:
print("=== State classification (raw model output) ===")
print(classification_report(Y_test_state, state_pred, target_names=CONFIG["appliances"], zero_division=0))

print("\n=== State classification (after dwell-time correction) ===")
print(classification_report(Y_test_state, state_pred_corrected, target_names=CONFIG["appliances"], zero_division=0))

for i, app in enumerate(CONFIG["appliances"]):
    p = precision_score(Y_test_state[:, i], state_pred_corrected[:, i], zero_division=0)
    r = recall_score(Y_test_state[:, i], state_pred_corrected[:, i], zero_division=0)
    f1 = f1_score(Y_test_state[:, i], state_pred_corrected[:, i], zero_division=0)
    print(f"{app:20s} | Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f}")

# --- Duration estimation: predicted active minutes vs ground truth, using the duty head ---
window_minutes = CONFIG["window_size"] * 6 / 60.0  # window duration in minutes
print("\n=== Total active-time estimation (test split) ===")
print(f"{'Appliance':20s} {'Predicted (min)':>16s} {'Ground truth (min)':>20s} {'Error (min)':>14s}")
for i, app in enumerate(CONFIG["appliances"]):
    pred_minutes = float(np.sum(duty_pred[:, i] * state_pred_corrected[:, i]) * window_minutes)
    true_minutes = float(np.sum(Y_test_duty[:, i]) * window_minutes)
    error = pred_minutes - true_minutes
    print(f"{app:20s} {pred_minutes:16.1f} {true_minutes:20.1f} {error:14.1f}")

=== State classification (raw model output) ===
                 precision    recall  f1-score   support

     dishwasher       0.37      0.99      0.54       948
washing machine       0.80      0.99      0.89      2798
         laptop       0.89      0.95      0.92     20429
         fridge       0.96      0.98      0.97     17180
  office lamp 1       0.45      0.91      0.60      5901

      micro avg       0.79      0.96      0.87     47256
      macro avg       0.69      0.96      0.78     47256
   weighted avg       0.84      0.96      0.89     47256
    samples avg       0.66      0.74      0.68     47256


=== State classification (after dwell-time correction) ===
                 precision    recall  f1-score   support

     dishwasher       0.39      0.99      0.56       948
washing machine       0.81      0.99      0.89      2798
         laptop       0.89      0.95      0.92     20429
         fridge       0.96      0.98      0.97     17180
  office lamp 1       0.45      0

## 19. Explainability Spot Checks
Manually inspect a true-positive and a true-negative prediction for one appliance to
confirm the model is distinguishing real activity from same-scale noise, not just
fitting spurious correlations.

In [ ]:
def spot_check(appliance, seq_raw, y_true, y_pred_prob, n_examples=1):
    idx = CONFIG["appliances"].index(appliance)
    true_labels = y_true[:, idx]
    pred_probs = y_pred_prob[:, idx]

    tp_candidates = np.where((true_labels == 1) & (pred_probs > 0.5))[0]
    tn_candidates = np.where((true_labels == 0) & (pred_probs < 0.5))[0]

    print(f"--- {appliance}: true positive example ---")
    for idx_ex in tp_candidates[:n_examples]:
        w = seq_raw[idx_ex]
        print(f"  mean power {w.mean():.1f}W, max power {w.max():.1f}W "
              f"-> predicted ON at {pred_probs[idx_ex]:.2f} confidence. Correct.")

    print(f"--- {appliance}: true negative example ---")
    for idx_ex in tn_candidates[:n_examples]:
        w = seq_raw[idx_ex]
        print(f"  mean power {w.mean():.1f}W, max power {w.max():.1f}W "
              f"-> predicted OFF at {pred_probs[idx_ex]:.2f} confidence. Correct.")

for app in CONFIG["appliances"]:
    spot_check(app, X_test_seq, Y_test_state, state_prob)
    print()

--- dishwasher: true positive example ---
  mean power 2093.9W, max power 2167.0W -> predicted ON at 0.60 confidence. Correct.
--- dishwasher: true negative example ---
  mean power 114.9W, max power 163.0W -> predicted OFF at 0.39 confidence. Correct.

--- washing machine: true positive example ---
  mean power 1143.5W, max power 2167.0W -> predicted ON at 0.99 confidence. Correct.
--- washing machine: true negative example ---
  mean power 162.7W, max power 186.0W -> predicted OFF at 0.48 confidence. Correct.

--- laptop: true positive example ---
  mean power 162.7W, max power 186.0W -> predicted ON at 0.91 confidence. Correct.
--- laptop: true negative example ---
  mean power 30.6W, max power 121.0W -> predicted OFF at 0.48 confidence. Correct.

--- fridge: true positive example ---
  mean power 162.7W, max power 186.0W -> predicted ON at 0.85 confidence. Correct.
--- fridge: true negative example ---
  mean power 95.9W, max power 163.0W -> predicted OFF at 0.46 confidence. Correc